# MiniGPT：从 bytes 到因果 logits

**适合读者**：刚学完 Attention、准备理解 decoder-only LM 前向的学习者。

**先修**：token ID、交叉熵、causal mask；需要 PyTorch 与 JAX，不需要 GPU。

运行前先按 [环境与内核说明](README.md) 准备依赖，并选择 `Python (about-llm)` 内核。

**路线**：UTF-8 bytes → inputs/targets → PyTorch logits/loss → 因果正反例 → JAX 接口对照。

**完成信号**：能画出 $[B,T] \to [B,T,V]$，说明 target shift，并区分接口对齐与权重数值 parity。

## 1. 先确认 tokenizer 的真实单位

ByteTokenizer 的 vocabulary 是 256 个 byte value，不是中文字符表。在运行前预测：字符串中的一个汉字会对应一个还是多个 token？

In [ ]:
from about_llm.from_scratch import ByteTokenizer

tokenizer = ByteTokenizer()
text = 'LLM 你好'
token_ids = tokenizer.encode(text)
assert tokenizer.decode(token_ids) == text
print('UTF-8 bytes:', token_ids)


## 2. 手算 shape，再运行 PyTorch

若输入 shape 是 $[B,T]$，logits 应是 $[B,T,V]$。这里 targets 向左移动一位，因此第 $t$ 个 input 学习预测原序列第 $t+1$ 个 byte。随机初始化 loss 只能检查数值和接口，不能评价语言质量。

In [ ]:
import torch

from about_llm.from_scratch.gpt_torch import GPTConfig, MiniGPT

torch.manual_seed(0)
config = GPTConfig(
    vocab_size=256, context_length=16, model_dim=32,
    num_heads=4, num_layers=2, mlp_ratio=2,
)
model = MiniGPT(config).eval()
tokens = torch.tensor([token_ids[:8]], dtype=torch.long)
inputs, targets = tokens[:, :-1], tokens[:, 1:]
assert inputs.shape == targets.shape == (1, tokens.shape[1] - 1)
assert torch.equal(inputs[:, 1:], targets[:, :-1])
for position, (source_id, target_id) in enumerate(
    zip(inputs[0].tolist(), targets[0].tolist(), strict=True)
):
    print(f'position {position}: input byte {source_id} -> target byte {target_id}')
logits, loss = model(inputs, targets)
print('PyTorch logits:', tuple(logits.shape), 'loss:', round(loss.item(), 4))
assert logits.shape == (*inputs.shape, config.vocab_size)

# 随机初始化的模型对 vocab_size 个 token 近似均匀猜测，理论 loss ≈ ln(vocab_size)。
import math

print('ln(vocab_size) =', round(math.log(config.vocab_size), 4))
# 这里只做观察，不做断言：初始化方案和随机种子都会让实测值偏离这个基准，
# 但偏离一个数量级通常意味着初始化或 loss 计算有问题。


## 3. 用一对反事实输入验证 causal 不变量

两条序列的前两个 token 相同、未来 token 不同。先预测前两个位置的 logits 是否应变化，再运行断言。

In [ ]:
first = torch.tensor([[1, 2, 3, 4]])
second = torch.tensor([[1, 2, 9, 10]])
first_logits, _ = model(first)
second_logits, _ = model(second)
torch.testing.assert_close(first_logits[:, :2], second_logits[:, :2])
print('Causality check passed: future tokens do not change past logits.')


## 4. 反过来修改过去，未来应当变化

上一格只检查“未来不能影响过去”。现在修改第一个 token，并观察最后位置 logits；这给 causal 方向补一个正对照。

In [ ]:
past_changed = torch.tensor([[8, 2, 3, 4]])
past_changed_logits, _ = model(past_changed)
last_position_delta = float((first_logits[:, -1] - past_changed_logits[:, -1]).abs().max())
print('max last-position logit change after editing the past:', round(last_position_delta, 6))
assert last_position_delta > 1e-6

## 5. 在 JAX 中检查同一接口契约

下面只比较 shape 和有限 loss。两边使用不同随机参数，而且实现细节可能不同，所以不能把“都能前向”写成数值 parity。真正的 parity 需要共享解析权重、norm、mask、loss reduction 和 optimizer contract。

In [ ]:
import jax
import jax.numpy as jnp

from about_llm.from_scratch.gpt_jax import JAXGPTConfig, cross_entropy_loss, forward, init_params

jax_config = JAXGPTConfig(
    vocab_size=256, context_length=16, model_dim=32,
    num_heads=4, num_layers=2, mlp_ratio=2,
)
params = init_params(jax.random.key(0), jax_config)
jax_logits = forward(params, jnp.asarray(inputs.numpy()), jax_config)
jax_loss = cross_entropy_loss(jax_logits, jnp.asarray(targets.numpy()))
print('JAX logits:', jax_logits.shape, 'loss:', round(float(jax_loss), 4))
assert jax_logits.shape == (*inputs.shape, jax_config.vocab_size)


## 6. 你现在证明了什么

已经证明：byte round-trip、两种实现的 logits shape，以及 PyTorch 路径中的 causal 正反对照。

尚未证明：训练能收敛、PyTorch/JAX 数值等价、生成有质量、checkpoint 可恢复或 GPU 性能。

**练习**：重新运行第 2 节，检查打印的每一行是否满足第 $t$ 个 target 等于原序列第 $t+1$ 个 byte；再把 `tokens` 换成 `tokenizer.encode('LLM 再见')[:8]`，预测并检查新 shape。

**答案脚手架**：第 2 节已经用 `assert` 和逐位置打印执行了这个对齐检查。target 第 $t$ 项来自原序列第 $t+1$ 个 token。

**扩展：训练与留出评估（不是本页 forward 验收）**：运行 [04_sft_sample_lifecycle.ipynb](04_sft_sample_lifecycle.ipynb)。它在固定 tiny batch 上执行 40 次 LoRA 更新，并单独报告一个未参加更新的样本；训练 loss 下降只说明能拟合该 batch，留出单例仍不代表模型泛化或语言质量。